Financial institutions process a huge number of transactions every day, making it difficult to manually identify fraudulent transactions. Since fraudulent transactions are extremely rare compared to legitimate ones, a model that simply achieves high accuracy may still fail to detect actual fraud. Therefore, the goal of this project is to develop a machine learning–based fraud detection system that can identify potentially fraudulent transactions while minimizing false positives and false negatives.

In [39]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import math as m
import statsmodels.formula.api as smf

import warnings
warnings.filterwarnings('ignore')


In [40]:
df = pd.read_csv("../Data/bs140513_032310.csv/bs140513_032310.csv")

In [41]:
df

,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
0,0,'C1093826151','4','M','28007','M348934600','28007','es_transportation',4.55,0
1,0,'C352968107','2','M','28007','M348934600','28007','es_transportation',39.68,0
2,0,'C2054744914','4','F','28007','M1823072687','28007','es_transportation',26.89,0
3,0,'C1760612790','3','M','28007','M348934600','28007','es_transportation',17.25,0
4,0,'C757503768','5','M','28007','M348934600','28007','es_transportation',35.72,0
...,...,...,...,...,...,...,...,...,...,...
594638,179,'C1753498738','3','F','28007','M1823072687','28007','es_transportation',20.53,0
594639,179,'C650108285','4','F','28007','M1823072687','28007','es_transportation',50.73,0
594640,179,'C123623130','2','F','28007','M349281107','28007','es_fashion',22.44,0
594641,179,'C1499363341','5','M','28007','M1823072687','28007','es_transportation',14.46,0


## BankSim Dataset — Variable Descriptions

| Variable | Description |
|---|---|
| `step` | Represents the time granularity of the simulation. Each step corresponds to one day of simulated commercial activity. |
| `customer` | An identifier for the customer involved in the transaction. It helps identify transactions belonging to the same customer. |
| `age` | Represents the customer's age category: `'0'` (18 years or younger), `'1'` (19–25), `'2'` (26–35), `'3'` (36–45), `'4'` (46–55), `'5'` (56–65), `'6'` (over 65), and `'U'` (unknown age). |
| `gender` | Represents the customer's gender category. The documented categories are `E` (Enterprise), `F` (Female), `M` (Male), and `U` (Unknown). |
| `zipcodeOri` | Represents the ZIP code associated with the customer's origin. |
| `merchant` | An identifier for the merchant involved in the transaction. |
| `zipMerchant` | Represents the ZIP code associated with the merchant's location. |
| `category` | Represents the merchant category associated with the purchased goods or services, such as transportation, food, restaurants, or fashion. |
| `amount` | Represents the monetary value of the transaction, measured in euros (€). |
| `fraud` | The binary target variable indicating whether a transaction is fraudulent: `0` = legitimate transaction and `1` = fraudulent transaction. |

### Key Observations

- Each row represents one transaction.
- A customer may have multiple transactions with different merchants.
- `customer` and `merchant` are identifiers, not numerical measurements.
- `age`, `gender`, and `category` are categorical features.
- `amount` represents the transaction value, while `fraud` is the target variable for our prediction task.

**Note:** The dataset is synthetic and simulates financial transactions. We will inspect the actual values and distributions of each column before deciding on data cleaning, encoding, or feature engineering.

In [42]:
df.columns

Index(['step', 'customer', 'age', 'gender', 'zipcodeOri', 'merchant',
       'zipMerchant', 'category', 'amount', 'fraud'],
      dtype='object')

In [43]:
df.shape

(594643, 10)

In [44]:
df.head()

,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
0,0,'C1093826151','4','M','28007','M348934600','28007','es_transportation',4.55,0
1,0,'C352968107','2','M','28007','M348934600','28007','es_transportation',39.68,0
2,0,'C2054744914','4','F','28007','M1823072687','28007','es_transportation',26.89,0
3,0,'C1760612790','3','M','28007','M348934600','28007','es_transportation',17.25,0
4,0,'C757503768','5','M','28007','M348934600','28007','es_transportation',35.72,0


In [45]:
df.tail()

,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
594638,179,'C1753498738','3','F','28007','M1823072687','28007','es_transportation',20.53,0
594639,179,'C650108285','4','F','28007','M1823072687','28007','es_transportation',50.73,0
594640,179,'C123623130','2','F','28007','M349281107','28007','es_fashion',22.44,0
594641,179,'C1499363341','5','M','28007','M1823072687','28007','es_transportation',14.46,0
594642,179,'C616528518','4','F','28007','M1823072687','28007','es_transportation',26.93,0


In [46]:
df.sample(10)

,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
192266,66,'C949319027','2','M','28007','M1823072687','28007','es_transportation',29.61,0
167001,58,'C198230909','4','F','28007','M348934600','28007','es_transportation',53.05,0
487485,151,'C49864446','5','M','28007','M348934600','28007','es_transportation',36.25,0
340621,110,'C165037464','1','F','28007','M85975013','28007','es_food',8.55,0
370334,118,'C642158646','6','F','28007','M348934600','28007','es_transportation',30.61,0
376316,120,'C1767103563','3','F','28007','M85975013','28007','es_food',20.50,0
256360,85,'C1338020899','3','M','28007','M78078399','28007','es_wellnessandbeauty',69.81,0
476633,148,'C1539370210','3','M','28007','M1400236507','28007','es_home',129.09,0
465869,145,'C1575842015','5','M','28007','M1823072687','28007','es_transportation',9.65,0
19039,7,'C358037591','3','M','28007','M348934600','28007','es_transportation',14.32,0


In [47]:
df.isnull().sum().sum()

np.int64(0)

In [48]:
df.duplicated().sum()

np.int64(0)

In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594643 entries, 0 to 594642
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   step         594643 non-null  int64  
 1   customer     594643 non-null  object 
 2   age          594643 non-null  object 
 3   gender       594643 non-null  object 
 4   zipcodeOri   594643 non-null  object 
 5   merchant     594643 non-null  object 
 6   zipMerchant  594643 non-null  object 
 7   category     594643 non-null  object 
 8   amount       594643 non-null  float64
 9   fraud        594643 non-null  int64  
dtypes: float64(1), int64(2), object(7)
memory usage: 45.4+ MB


In [50]:
for col in df.columns:
    print(f"{col} : {df[col].unique()} Unique values\n")

step : [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179] Unique values

customer : ["'C1093826151'" "'C352968107'" "'C2054744914'" ... "'C1014783484'"
 "'C849065220'" "'C1809646275'"] Unique values

age : ["'4'" "'2'" "'3'" "'5'" "'1'" "'6'" "'U'" "'0'"] Unique values

gender : ["'M'" "'F'" "'E'" "'U'"] Unique values


In [51]:
df.dtypes

step             int64
customer        object
age             object
gender          object
zipcodeOri      object
merchant        object
zipMerchant     object
category        object
amount         float64
fraud            int64
dtype: object

In [52]:

df.duplicated().sum()

np.int64(0)

In [53]:
for col in df.columns:
    print(col, ":", df[col].nunique())

step : 180
customer : 4112
age : 8
gender : 4
zipcodeOri : 1
merchant : 50
zipMerchant : 1
category : 15
amount : 23767
fraud : 2


In [54]:
df[['step', 'amount']].describe()

,step,amount
count,594643.000000,594643.000000
mean,94.986827,37.890135
std,51.053632,111.402831
min,0.000000,0.000000
25%,52.000000,13.740000
50%,97.000000,26.900000
75%,139.000000,42.540000
max,179.000000,8329.960000


In [55]:
print("Zero amounts:", (df['amount'] == 0).sum())
print("Negative amounts:", (df['amount'] < 0).sum())
print("Positive amounts:", (df['amount'] > 0).sum())

Zero amounts: 52
Negative amounts: 0
Positive amounts: 594591


In [ ]:
df['fraud'].value_counts(dropna=False)

fraud
0    587443
1      7200
Name: count, dtype: int64